# NCU Spotlight — PAN1/PAN2 Frame Alignment & PSF Analysis

This notebook processes telescope "Spotlight" TIFF captures into a
stacked, star-sharpness (PSF) analysis pipeline:

1. **Convert** raw `.tif` frames to `.fits` files.
2. **Align & stack** each `PAN1`/`PAN2` frame pair (two experimental
   approaches are kept below for reference — a hybrid catalog+phase
   method, and a simpler correlation-only method).
3. **Measure PSF (FWHM)** of bright stars in each stacked image by
   fitting a 2D Gaussian, and summarize the sharpness distribution
   across all pairs.

Each code cell below has English comments explaining what it does.


## Step 1 — Convert TIFF to FITS

In [14]:
# ============================================================
# Step 1: Batch-convert raw TIFF images into FITS format
# ============================================================
# Astronomy analysis tools (astropy, sep, etc.) expect FITS files,
# so we first walk the raw data folder, find every .tif/.tiff file
# (skipping any already-converted "fits_outputs" folder and any
# hidden files/folders), and convert each one into a matching .fits
# file while preserving the original sub-folder structure.

from pathlib import Path
import tifffile
from astropy.io import fits

# Source folder containing the raw TIFF captures.
src_root = Path('./NCU Spotlight/raw_data')
# Destination folder for the converted FITS files (mirrors src_root's layout).
dst_root = src_root / 'fits_outputs'
dst_root.mkdir(parents=True, exist_ok=True)

# Collect all TIFF files under src_root, excluding:
#   - anything already inside the fits_outputs destination folder
#   - hidden files/folders (names starting with '.')
all_tif_files = [
    p for p in src_root.rglob('*')
    if p.is_file()
    and p.suffix.lower() in ('.tif', '.tiff')
    and 'fits_outputs' not in p.relative_to(src_root).parts
    and not any(part.startswith('.') for part in p.relative_to(src_root).parts)
]

converted = 0
failed = 0

# Convert each TIFF to FITS, keeping the same relative sub-path.
for tif_path in sorted(all_tif_files):
    out_path = dst_root / tif_path.relative_to(src_root).parent / f'{tif_path.stem}.fits'
    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        image_data = tifffile.imread(str(tif_path))
        # overwrite=True lets us re-run this cell safely.
        fits.PrimaryHDU(image_data).writeto(str(out_path), overwrite=True)
        print(f'CONVERTED: {tif_path.relative_to(src_root)} -> {out_path.relative_to(src_root)}')
        converted += 1
    except Exception as exc:
        print(f'FAILED: {tif_path.relative_to(src_root)} ({exc})')
        failed += 1

print(f'Summary: {converted} converted, {failed} failed')


CONVERTED: 20260525_PAN1.tif -> fits_outputs/20260525_PAN1.fits
CONVERTED: 20260525_PAN2.tif -> fits_outputs/20260525_PAN2.fits
CONVERTED: 20260602_PAN1.tif -> fits_outputs/20260602_PAN1.fits
CONVERTED: 20260602_PAN2.tif -> fits_outputs/20260602_PAN2.fits
CONVERTED: 20260603_PAN1.tif -> fits_outputs/20260603_PAN1.fits
CONVERTED: 20260603_PAN2.tif -> fits_outputs/20260603_PAN2.fits
CONVERTED: 20260622_PAN1.tif -> fits_outputs/20260622_PAN1.fits
CONVERTED: 20260622_PAN2.tif -> fits_outputs/20260622_PAN2.fits
CONVERTED: 20260623_PAN2.tif -> fits_outputs/20260623_PAN2.fits
CONVERTED: 20260702_PAN1.tif -> fits_outputs/20260702_PAN1.fits
CONVERTED: 20260702_PAN2.tif -> fits_outputs/20260702_PAN2.fits
CONVERTED: 20260707_PAN1.tif -> fits_outputs/20260707_PAN1.fits
CONVERTED: 20260707_PAN2.tif -> fits_outputs/20260707_PAN2.fits
CONVERTED: 20260709_PAN1.tif -> fits_outputs/20260709_PAN1.fits
CONVERTED: 20260709_PAN2.tif -> fits_outputs/20260709_PAN2.fits
CONVERTED: 20260716_PAN1.tif -> fits_out

## Step 2a — Hybrid approach (SEP catalog + phase correlation)

Earlier version of the alignment pipeline: tries three shift-estimation methods (bright-star catalog matching, phase correlation, and a fixed fallback), scores each by stack sharpness, and keeps the best one.

In [43]:
# ============================================================
# Step 2a (early approach): Hybrid shift estimation + stacking
# ============================================================
# For each PAN1/PAN2 frame pair, estimate the pixel shift needed to
# align PAN2 onto PAN1 using THREE candidate methods, then pick
# whichever gives the sharpest stacked result:
#   1. "catalog"  - match bright-star positions detected by SEP
#   2. "phase"    - classic phase cross-correlation
#   3. "fallback" - a fixed default shift, used when the others fail
# The best-scoring shift is refined on a fine local grid, the two
# frames are averaged (stacked), and the result is written out as a
# FITS file plus a 3-panel (PAN1 / PAN2 / STACK) page in a PDF report.
#
# NOTE: this is the earlier, more complex version of the pipeline.
# See the "Using only correlation" section below for the simplified,
# correlation-only version that also adds PSF (star sharpness) fitting.

from pathlib import Path
import re
import numpy as np
import sep
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.ndimage import shift as ndi_shift
from scipy.spatial import cKDTree
from skimage.registration import phase_cross_correlation
from astropy.io import fits

workspace_root = Path('/Users/chyan/Desktop/NCU Spotlight')

# Paths
# Try several plausible locations for the FITS input/output folders,
# so the notebook still works whether it's run from the workspace
# root or from inside the "NCU Spotlight" project folder.
fits_candidates = [
    workspace_root / 'fits_outputs',
    workspace_root / 'NCU Spotlight' / 'fits_outputs',
    Path.cwd() / 'fits_outputs',
    Path.cwd() / 'NCU Spotlight' / 'fits_outputs',
]
stack_candidates = [
    workspace_root / 'fits_stacked',
    workspace_root / 'NCU Spotlight' / 'fits_stacked',
    Path.cwd() / 'fits_stacked',
    Path.cwd() / 'NCU Spotlight' / 'fits_stacked',
]
fits_root = next((p for p in fits_candidates if p.exists()), fits_candidates[0])
stack_root = next((p for p in stack_candidates if p.exists()), stack_candidates[0])
stack_root.mkdir(parents=True, exist_ok=True)
pdf_path = stack_root / 'pan_pair_triptych.pdf'

# Parameters
fallback_shift = (-0.5, -0.5)   # default (dy, dx) shift used when estimation fails
n_bright = 30                   # number of brightest stars to use for catalog matching
match_radius = 1.0              # max pixel distance to count two stars as "matched"
min_matches_ok = 2              # minimum matched stars required to trust a non-fallback shift
max_rms_ok = 1.0                # maximum matching RMS (pixels) required to trust a shift


def center_crop(arr, ny, nx):
    """Crop a 2D array to (ny, nx) around its center."""
    y0 = max(0, (arr.shape[0] - ny) // 2)
    x0 = max(0, (arr.shape[1] - nx) // 2)
    return arr[y0:y0 + ny, x0:x0 + nx]


def detect_sources(image, thresh=4.0, minarea=5):
    """Run SEP source extraction and return (y, x) star coordinates and flux."""
    finite = np.isfinite(image)
    if np.count_nonzero(finite) < 100:
        return np.empty((0, 2)), np.empty((0,))

    # Fill non-finite pixels with the image median so SEP can process the array.
    fill = float(np.nanmedian(image[finite]))
    img = np.where(finite, image, fill).astype(np.float32)
    bkg = sep.Background(img)
    obj = sep.extract(img - bkg, thresh=thresh, err=bkg.globalrms, minarea=minarea)

    if len(obj) == 0:
        return np.empty((0, 2)), np.empty((0,))

    coords = np.vstack([obj['y'], obj['x']]).T.astype(np.float64)
    flux = np.asarray(obj['flux'], dtype=np.float64)
    return coords, flux


def match_stats(coords1, coords2, shift_yx, radius=1.0):
    """Count one-to-one star matches between coords1 and shifted coords2,
    and return (number of matches, RMS matching distance in pixels)."""
    if len(coords1) == 0 or len(coords2) == 0:
        return 0, np.inf

    tree = cKDTree(coords2)
    query = np.column_stack([coords1[:, 0] - shift_yx[0], coords1[:, 1] - shift_yx[1]])
    dist, nn = tree.query(query, distance_upper_bound=radius)
    good = np.isfinite(dist) & (dist < radius)
    if np.count_nonzero(good) == 0:
        return 0, np.inf

    # Keep one-to-one matches by nearest distance.
    dist_good = dist[good]
    nn_good = nn[good]
    order = np.argsort(dist_good)
    used = set()
    keep = []
    for k in order:
        j = int(nn_good[k])
        if j in used:
            continue
        used.add(j)
        keep.append(float(dist_good[k]))

    if not keep:
        return 0, np.inf
    keep = np.asarray(keep, dtype=np.float64)
    return len(keep), float(np.sqrt(np.mean(keep**2)))


def stack_with_shift(data1, data2, shift_yx):
    """Shift data2 by shift_yx and average it with data1 (only where both are valid)."""
    data2_aligned = ndi_shift(data2, shift=shift_yx, order=3, mode='constant', cval=np.nan)
    valid2 = ndi_shift(np.ones_like(data2, dtype=np.float32), shift=shift_yx, order=0, mode='constant', cval=0.0) > 0.5
    return np.where(valid2, 0.5 * (data1 + data2_aligned), np.nan)


def sharpness_score(stacked, ref_coords):
    """Score how sharp/peaked the stacked image is around reference star
    positions (higher = sharper stack = better alignment)."""
    finite = np.isfinite(stacked)
    if np.count_nonzero(finite) < 50:
        return -np.inf

    if len(ref_coords) == 0:
        # No reference stars: fall back to a global contrast-based score.
        med = np.nanmedian(stacked[finite])
        std = np.nanstd(stacked[finite]) + 1e-6
        return float((np.nanmax(stacked[finite]) - med) / std)

    vals = []
    for y, x in ref_coords:
        yi = int(round(y))
        xi = int(round(x))
        if yi < 3 or xi < 3 or yi >= stacked.shape[0] - 3 or xi >= stacked.shape[1] - 3:
            continue
        patch = stacked[yi - 2:yi + 3, xi - 2:xi + 3]
        ok = np.isfinite(patch)
        if np.count_nonzero(ok) < 20:
            continue
        p = patch[ok]
        b = np.median(p)
        peak = np.max(p) - b
        flux = np.sum(np.clip(p - b, 0.0, None)) + 1e-6
        # peak / flux is high for a sharp, concentrated PSF and low for a blurred one.
        vals.append(peak / flux)

    return float(np.median(vals)) if vals else -np.inf


def estimate_shift_catalog(coords1, flux1, coords2, flux2):
    """Estimate the global (dy, dx) shift from the most common pairwise
    offset between the brightest stars in each frame (mode of a coarse
    2D histogram of star-to-star offsets)."""
    if len(coords1) == 0 or len(coords2) == 0:
        return fallback_shift

    c1 = coords1[np.argsort(flux1)[-n_bright:]]
    c2 = coords2[np.argsort(flux2)[-n_bright:]]

    dy = c1[:, None, 0] - c2[None, :, 0]
    dx = c1[:, None, 1] - c2[None, :, 1]
    deltas = np.column_stack([dy.ravel(), dx.ravel()])

    # Use 2D histogram mode as robust global shift estimate.
    q = np.round(deltas * 4.0) / 4.0
    uniq, cnt = np.unique(q, axis=0, return_counts=True)
    return (float(uniq[np.argmax(cnt), 0]), float(uniq[np.argmax(cnt), 1]))


def estimate_shift_phase(data1, data2):
    """Estimate the global (dy, dx) shift using sub-pixel phase cross-correlation."""
    finite1 = np.isfinite(data1)
    finite2 = np.isfinite(data2)
    if np.count_nonzero(finite1) < 50 or np.count_nonzero(finite2) < 50:
        return fallback_shift

    fill1 = float(np.nanmedian(data1[finite1]))
    fill2 = float(np.nanmedian(data2[finite2]))
    img1 = np.where(finite1, data1, fill1).astype(np.float32)
    img2 = np.where(finite2, data2, fill2).astype(np.float32)
    img1 -= np.median(img1)
    img2 -= np.median(img2)

    s, _, _ = phase_cross_correlation(img1, img2, upsample_factor=20)
    return (float(s[0]), float(s[1]))


def refine_shift(data1, data2, coords1, coords2, flux1, base_shift):
    """Locally refine a candidate shift on a +/-0.6px grid (0.1px step),
    scoring each candidate by stack sharpness, match count, and match RMS.
    Returns the best candidate as a dict with 'shift', 'n_match', 'rms',
    'stacked', and 'score'."""
    if len(coords1) > 0:
        ref_coords = coords1[np.argsort(flux1)[-12:]]
    else:
        ref_coords = np.empty((0, 2))

    dy0, dx0 = base_shift
    dy_grid = np.arange(dy0 - 0.6, dy0 + 0.6 + 1e-9, 0.1)
    dx_grid = np.arange(dx0 - 0.6, dx0 + 0.6 + 1e-9, 0.1)

    best = None
    for dy in dy_grid:
        for dx in dx_grid:
            s = (float(dy), float(dx))
            n_match, rms = match_stats(coords1, coords2, s, radius=match_radius)
            st = stack_with_shift(data1, data2, s)
            sharp = sharpness_score(st, ref_coords)
            # Combine sharpness with a small bonus for more matches and a
            # small penalty for higher RMS, to break ties sensibly.
            score = sharp + 0.01 * n_match - 0.02 * (rms if np.isfinite(rms) else 10.0)
            if (best is None) or (score > best['score']):
                best = {'shift': s, 'n_match': n_match, 'rms': rms, 'stacked': st, 'score': score}

    return best


def robust_limits(arr):
    """Return (vmin, vmax) display limits using the 5th/99.5th percentiles,
    for robust (outlier-resistant) image display scaling."""
    finite = np.isfinite(arr)
    if np.count_nonzero(finite) < 10:
        return 0.0, 1.0
    vmin, vmax = np.nanpercentile(arr[finite], [5, 99.5])
    if (not np.isfinite(vmin)) or (not np.isfinite(vmax)) or (vmax <= vmin):
        return float(np.nanmin(arr[finite])), float(np.nanmax(arr[finite]))
    return float(vmin), float(vmax)


# Find every PAN1 frame; its matching PAN2 frame is derived by name substitution.
pan1_files = sorted(fits_root.rglob('*PAN1.fits')) if fits_root.exists() else []
combined = 0
missing = 0
failed = 0
cropped = 0
fallback_used = 0

with PdfPages(pdf_path) as pdf:
    for pan1_path in pan1_files:
        pan2_name = re.sub(r'PAN1\.fits$', 'PAN2.fits', pan1_path.name, flags=re.IGNORECASE)
        pan2_path = pan1_path.with_name(pan2_name)

        if not pan2_path.exists():
            print(f'MISSING PAIR: {pan1_path.relative_to(fits_root)} (no PAN2)')
            missing += 1
            continue

        try:
            data1 = fits.getdata(pan1_path).astype(np.float32)
            data2 = fits.getdata(pan2_path).astype(np.float32)
            if data1.ndim < 2 or data2.ndim < 2:
                print(f'INVALID DIMENSION: {pan1_path.name} or {pan2_path.name}')
                failed += 1
                continue

            # If PAN1/PAN2 have different sizes, crop both to their common
            # center-aligned overlap region before comparing/stacking.
            ny = min(data1.shape[0], data2.shape[0])
            nx = min(data1.shape[1], data2.shape[1])
            if data1.shape[:2] != (ny, nx) or data2.shape[:2] != (ny, nx):
                print(f'CROP TO OVERLAP: {pan1_path.name} {data1.shape[:2]} & {pan2_path.name} {data2.shape[:2]} -> {(ny, nx)}')
                cropped += 1

            data1_crop = center_crop(data1, ny, nx)
            data2_crop = center_crop(data2, ny, nx)

            coords1, flux1 = detect_sources(data1_crop)
            coords2, flux2 = detect_sources(data2_crop)

            # Three candidates: catalog / phase / fallback, each with local refinement.
            cands = {
                'catalog': estimate_shift_catalog(coords1, flux1, coords2, flux2),
                'phase': estimate_shift_phase(data1_crop, data2_crop),
                'fallback': fallback_shift,
            }

            evals = {name: refine_shift(data1_crop, data2_crop, coords1, coords2, flux1, s) for name, s in cands.items()}
            method = max(evals, key=lambda k: evals[k]['score'])
            best = evals[method]

            # Require minimum matching quality unless fallback is best by score.
            if method != 'fallback':
                ok = best['n_match'] >= min_matches_ok and np.isfinite(best['rms']) and best['rms'] <= max_rms_ok
                if not ok:
                    method = 'fallback'
                    best = evals['fallback']
                    fallback_used += 1
            else:
                fallback_used += 1

            shift_yx = best['shift']
            n_match = best['n_match']
            rms = best['rms']
            stacked = best['stacked']
            score_gain = best['score'] - evals['fallback']['score']

            out_name = re.sub(r'PAN1\.fits$', 'PAN_stack.fits', pan1_path.name, flags=re.IGNORECASE)
            out_path = stack_root / pan1_path.relative_to(fits_root).parent / out_name
            out_path.parent.mkdir(parents=True, exist_ok=True)

            # Copy the PAN1 header and record alignment/stacking metadata.
            hdr = fits.getheader(pan1_path)
            hdr['HISTORY'] = 'PAN2 aligned by simplified hybrid SEP+phase estimator'
            hdr['SHIFT_Y'] = (shift_yx[0], 'Shift applied to PAN2 along Y (pixel)')
            hdr['SHIFT_X'] = (shift_yx[1], 'Shift applied to PAN2 along X (pixel)')
            hdr['SHFMETH'] = (method, 'Selected shift method')
            hdr['NMATCH'] = (n_match, 'One-to-one matched stars')
            hdr['RMSSHF'] = (float(rms) if np.isfinite(rms) else -1.0, 'Residual RMS, -1 if invalid')
            hdr['SCGAIN'] = (float(score_gain), 'Score gain over fallback')
            hdr['STACKED'] = ('PAN1+PAN2', 'Two-frame stacked product')

            fits.PrimaryHDU(stacked.astype(np.float32), header=hdr).writeto(out_path, overwrite=True)

            # PDF triptych: PAN1, PAN2, STACK
            p1_vmin, p1_vmax = robust_limits(data1_crop)
            p2_vmin, p2_vmax = robust_limits(data2_crop)
            st_vmin, st_vmax = robust_limits(stacked)

            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(data1_crop, origin='lower', cmap='gray', vmin=p1_vmin, vmax=p1_vmax)
            axes[0].set_title('PAN1')
            axes[1].imshow(data2_crop, origin='lower', cmap='gray', vmin=p2_vmin, vmax=p2_vmax)
            axes[1].set_title('PAN2')
            axes[2].imshow(stacked, origin='lower', cmap='gray', vmin=st_vmin, vmax=st_vmax)
            axes[2].set_title('STACK')
            for ax in axes:
                ax.set_xlabel('X')
                ax.set_ylabel('Y')

            fig.suptitle(f"{pan1_path.stem} | shift=(dy={shift_yx[0]:.3f}, dx={shift_yx[1]:.3f}), method={method}, matched={n_match}, rms={rms if np.isfinite(rms) else float('nan'):.3f}")
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

            print(f'STACKED: {pan1_path.relative_to(fits_root)} + {pan2_path.relative_to(fits_root)} | shift={shift_yx}, method={method}, matched={n_match}, rms={rms:.3f} -> {out_path.relative_to(stack_root.parent)}')
            combined += 1

        except Exception as exc:
            print(f'FAILED: {pan1_path.name} ({exc})')
            failed += 1

print(f'PDF: {pdf_path}')
print(f'Summary: {combined} stacked, {missing} missing pairs, {failed} failed, {cropped} overlap-cropped, {fallback_used} fallback-used')


CROP TO OVERLAP: 20260525_PAN1.fits (67, 70) & 20260525_PAN2.fits (68, 71) -> (67, 70)
STACKED: 20260525_PAN1.fits + 20260525_PAN2.fits | shift=(-1.0, 0.10000000000000098), method=fallback, matched=0, rms=inf -> fits_stacked/20260525_PAN_stack.fits
CROP TO OVERLAP: 20260602_PAN1.fits (63, 62) & 20260602_PAN2.fits (63, 56) -> (63, 56)
STACKED: 20260602_PAN1.fits + 20260602_PAN2.fits | shift=(5.000000190734861, 1.9500000476837158), method=phase, matched=3, rms=0.514 -> fits_stacked/20260602_PAN_stack.fits
CROP TO OVERLAP: 20260603_PAN1.fits (70, 62) & 20260603_PAN2.fits (69, 58) -> (69, 58)
STACKED: 20260603_PAN1.fits + 20260603_PAN2.fits | shift=(-1.149999976158142, 0.800000023841858), method=phase, matched=3, rms=0.311 -> fits_stacked/20260603_PAN_stack.fits
CROP TO OVERLAP: 20260622_PAN1.fits (59, 52) & 20260622_PAN2.fits (49, 40) -> (49, 40)
STACKED: 20260622_PAN1.fits + 20260622_PAN2.fits | shift=(2.7, -1.9999999999999991), method=catalog, matched=2, rms=0.594 -> fits_stacked/202606

## Step 2b/3 — Using only correlation (refined approach)

Simplified alignment: phase correlation only (no star-catalog matching), followed by bright-star PSF (Gaussian FWHM) fitting and a summary distribution plot. This is the version used for the final PSF analysis.

In [21]:
# ============================================================
# Step 3: Correlation-only alignment + PSF (star sharpness) fitting
# ============================================================
# Simplified/refined pipeline compared to the hybrid version above:
#   1. Align PAN2 onto PAN1 using ONLY phase cross-correlation
#      (coarse FFT-based estimate, then a local fine grid search
#      that maximizes normalized image correlation).
#   2. Stack (average) the aligned pair.
#   3. Detect bright stars in the stack with SEP and fit a 2D
#      elliptical Gaussian PSF to each one, to measure its FWHM
#      (a proxy for image sharpness / seeing quality). We now fit ALL
#      detected stars (by default) and collect their FWHMs for the
#      final distribution histogram.
#   4. Save the stacked FITS file (with alignment/PSF metadata),
#      a per-pair PDF page (PAN1 / PAN2 / STACK+labels / fit profiles),
#      and a final histogram of FWHM across all processed pairs.

from pathlib import Path
import re
import numpy as np
import sep
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.ndimage import shift as ndi_shift
from scipy.optimize import curve_fit
from skimage.registration import phase_cross_correlation
from astropy.io import fits

workspace_root = Path('/Users/chyan/Desktop/NCU Spotlight')

# Correlation-only shift parameters.
fallback_shift = (-0.5, -0.5)   # default (dy, dx) shift if correlation estimation fails
upsample_factor = 20            # sub-pixel precision factor for the coarse FFT phase correlation
fine_half_range = 0.6           # +/- range (pixels) for the local fine-grid search
fine_step = 0.1                 # step size (pixels) for the local fine-grid search

# PSF measurement/label parameters.
sep_thresh_sigma = 3.5          # SEP detection threshold, in units of background RMS
sep_minarea = 5                 # minimum pixel area for a SEP detection
# The pipeline will measure all detected stars by default; use max_labels
# to put a hard cap if you want to bound runtime/memory (None = no cap).
max_labels = None

# PSF outlier-cleaning parameters for the final distribution plot.
outlier_iqr_k = 1.5             # IQR multiplier used to define outlier bounds
min_points_for_iqr = 8          # need at least this many FWHM values to apply IQR filtering


def first_existing(candidates):
    """Return the first path in `candidates` that already exists, else the first one."""
    return next((p for p in candidates if p.exists()), candidates[0])


def get_roots():
    """Resolve the FITS input folder and the stacked-output folder,
    trying several plausible working-directory layouts."""
    fits_root = first_existing([
        workspace_root / 'fits_outputs',
        workspace_root / 'NCU Spotlight' / 'fits_outputs',
        Path.cwd() / 'fits_outputs',
        Path.cwd() / 'NCU Spotlight' / 'fits_outputs',
    ])
    stack_root = first_existing([
        workspace_root / 'fits_stacked',
        workspace_root / 'NCU Spotlight' / 'fits_stacked',
        Path.cwd() / 'fits_stacked',
        Path.cwd() / 'NCU Spotlight' / 'fits_stacked',
    ])
    stack_root.mkdir(parents=True, exist_ok=True)
    return fits_root, stack_root


def center_crop(arr, ny, nx):
    """Crop a 2D array to (ny, nx) around its center."""
    y0 = max(0, (arr.shape[0] - ny) // 2)
    x0 = max(0, (arr.shape[1] - nx) // 2)
    return arr[y0:y0 + ny, x0:x0 + nx]


def crop_pair_to_common_center(data1, data2):
    """Center-crop both frames of a pair down to their common (smallest) shape."""
    ny = min(data1.shape[0], data2.shape[0])
    nx = min(data1.shape[1], data2.shape[1])
    return center_crop(data1, ny, nx), center_crop(data2, ny, nx)


def preprocess_for_corr(image):
    """Fill non-finite pixels with the median and standardize (zero mean,
    unit std) an image so it's ready for cross-correlation."""
    finite = np.isfinite(image)
    if np.count_nonzero(finite) < 50:
        return None

    fill = float(np.nanmedian(image[finite]))
    img = np.where(finite, image, fill).astype(np.float32)
    img -= np.median(img)
    std = np.std(img)
    if std > 0:
        img /= std
    return img


def corr_score(ref_img, mov_img, shift_yx):
    """Normalized cross-correlation between ref_img and mov_img shifted by shift_yx
    (only over pixels valid in both images). Higher is a better alignment."""
    moved = ndi_shift(mov_img, shift=shift_yx, order=3, mode='constant', cval=np.nan)
    valid = np.isfinite(ref_img) & np.isfinite(moved)
    if np.count_nonzero(valid) < 100:
        return -np.inf

    a = ref_img[valid].astype(np.float64)
    b = moved[valid].astype(np.float64)
    a -= np.mean(a)
    b -= np.mean(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b) + 1e-12
    return float(np.dot(a, b) / denom)


def estimate_shift_correlation(data1, data2):
    """Estimate the (dy, dx) shift of data2 relative to data1:
    first a coarse FFT-based phase cross-correlation, then a local
    fine grid search that maximizes normalized correlation.
    Returns (best_shift, best_correlation_score, phase_correlation_error)."""
    ref_img = preprocess_for_corr(data1)
    mov_img = preprocess_for_corr(data2)
    if ref_img is None or mov_img is None:
        return fallback_shift, -np.inf, np.nan

    coarse, phase_err, _ = phase_cross_correlation(ref_img, mov_img, upsample_factor=upsample_factor)
    dy0, dx0 = float(coarse[0]), float(coarse[1])

    dy_grid = np.arange(dy0 - fine_half_range, dy0 + fine_half_range + 1e-9, fine_step)
    dx_grid = np.arange(dx0 - fine_half_range, dx0 + fine_half_range + 1e-9, fine_step)

    best_shift = (dy0, dx0)
    best_corr = -np.inf
    for dy in dy_grid:
        for dx in dx_grid:
            s = (float(dy), float(dx))
            c = corr_score(ref_img, mov_img, s)
            if c > best_corr:
                best_corr = c
                best_shift = s

    return best_shift, best_corr, float(phase_err)


def stack_pair(data1, data2, shift_yx):
    """Shift data2 by shift_yx and average it with data1 (only where both are valid)."""
    moved = ndi_shift(data2, shift=shift_yx, order=3, mode='constant', cval=np.nan)
    valid = ndi_shift(np.ones_like(data2, dtype=np.float32), shift=shift_yx, order=0, mode='constant', cval=0.0) > 0.5
    return np.where(valid, 0.5 * (data1 + moved), np.nan)


def image_limits(arr):
    """Return (vmin, vmax) display limits using the 5th/99.5th percentiles,
    for robust (outlier-resistant) image display scaling."""
    finite = np.isfinite(arr)
    if np.count_nonzero(finite) < 10:
        return 0.0, 1.0
    vmin, vmax = np.nanpercentile(arr[finite], [5, 99.5])
    if (not np.isfinite(vmin)) or (not np.isfinite(vmax)) or (vmax <= vmin):
        return float(np.nanmin(arr[finite])), float(np.nanmax(arr[finite]))
    return float(vmin), float(vmax)


def gaussian2d_model(xy, amp, x0, y0, sx, sy, theta, offset):
    """2D elliptical Gaussian model (flattened) used to fit a star's PSF.
    amp: peak amplitude, (x0, y0): center, sx/sy: sigma along the ellipse
    axes, theta: rotation angle, offset: background level."""
    x, y = xy
    ct, st = np.cos(theta), np.sin(theta)
    a = (ct**2) / (2.0 * sx**2) + (st**2) / (2.0 * sy**2)
    b = -np.sin(2.0 * theta) / (4.0 * sx**2) + np.sin(2.0 * theta) / (4.0 * sy**2)
    c = (st**2) / (2.0 * sx**2) + (ct**2) / (2.0 * sy**2)
    z = amp * np.exp(-(a * (x - x0) ** 2 + 2.0 * b * (x - x0) * (y - y0) + c * (y - y0) ** 2)) + offset
    return z.ravel()


def fit_psf_on_patch(image, x, y, half_size=6, hr_scale=1):
    """Fit a 2D Gaussian PSF to a small patch centered near (x, y).
    If `hr_scale>1` a higher-resolution model grid will be returned under
    the key `model_hr` for plotting (otherwise `model_hr` is None).
    Returns a dict with the fitted center, FWHM (pixels), the patch,
    the fitted model image (same resolution as `patch`), and the local
    (patch-relative) center; or None if the fit isn't possible."""
    ny, nx = image.shape
    xi, yi = int(round(x)), int(round(y))
    x1, x2 = max(0, xi - half_size), min(nx, xi + half_size + 1)
    y1, y2 = max(0, yi - half_size), min(ny, yi + half_size + 1)

    patch = image[y1:y2, x1:x2]
    if patch.size < 25:
        return None

    valid = np.isfinite(patch)
    if np.count_nonzero(valid) < 25:
        return None

    z = patch.astype(np.float64)
    yy, xx = np.indices(z.shape, dtype=np.float64)

    # Initial parameter guess and bounds for the Gaussian fit.
    z_med = np.nanmedian(z)
    z_peak = np.nanmax(z)
    amp0 = max(z_peak - z_med, 1e-3)
    p0 = [amp0, float(x - x1), float(y - y1), 1.5, 1.5, 0.0, z_med]
    lower = [0.0, 0.0, 0.0, 0.3, 0.3, -np.pi / 2.0, -np.inf]
    upper = [np.inf, z.shape[1] - 1.0, z.shape[0] - 1.0, 8.0, 8.0, np.pi / 2.0, np.inf]

    try:
        popt, _ = curve_fit(
            gaussian2d_model,
            (xx[valid], yy[valid]),
            z[valid].ravel(),
            p0=p0,
            bounds=(lower, upper),
            maxfev=10000,
        )
    except Exception:
        return None

    _, x0, y0, sx, sy, _, _ = popt
    sx = float(abs(sx))
    sy = float(abs(sy))
    # Convert the (possibly elliptical) sigma to an equivalent circular
    # sigma, then to FWHM using the standard Gaussian conversion factor.
    sigma = np.sqrt(0.5 * (sx**2 + sy**2))
    fwhm = 2.354820045 * sigma

    # low-res model (same grid as the data patch)
    model = gaussian2d_model((xx, yy), *popt).reshape(z.shape)

    model_hr = None
    if hr_scale is not None and int(hr_scale) > 1:
        hr = int(hr_scale)
        x_hr = np.linspace(0, z.shape[1] - 1, z.shape[1] * hr)
        y_hr = np.linspace(0, z.shape[0] - 1, z.shape[0] * hr)
        xx_hr, yy_hr = np.meshgrid(x_hr, y_hr)
        model_hr = gaussian2d_model((xx_hr, yy_hr), *popt).reshape((z.shape[0] * hr, z.shape[1] * hr))

    return {
        'x': x1 + float(x0),
        'y': y1 + float(y0),
        'fwhm': float(fwhm),
        'patch': z,
        'model': model,
        'model_hr': model_hr,
        'hr_scale': int(hr_scale) if hr_scale is not None else 1,
        'x0_local': float(x0),
        'y0_local': float(y0),
    }


def measure_psf_for_bright_stars(stacked, measure_all=True, max_stars=None, half_size=6):
    """Detect stars in the stacked image with SEP and fit a Gaussian PSF to
    each detected object. By default `measure_all=True` so all detected
    objects are attempted; use `max_stars` to cap the number if needed.
    Returns a list of per-star PSF fit dicts (with 'flux' added).
    """
    finite = np.isfinite(stacked)
    if np.count_nonzero(finite) < 100:
        return []

    fill = float(np.nanmedian(stacked[finite]))
    img = np.where(finite, stacked, fill).astype(np.float32)
    bkg = sep.Background(img)
    sub = img - bkg

    objects = sep.extract(sub, thresh=sep_thresh_sigma, err=bkg.globalrms, minarea=sep_minarea)
    if len(objects) == 0:
        return []

    objs = list(objects)
    if not measure_all:
        # fallback to original "bright only" behaviour if requested
        flux = np.asarray(objects['flux'], dtype=np.float64)
        flux_cut = np.quantile(flux, 0.7)
        objs = [o for o in objs if float(o['flux']) >= float(flux_cut)]
        objs = sorted(objs, key=lambda o: float(o['flux']), reverse=True)[:max_labels]
    else:
        if max_stars is not None and len(objs) > max_stars:
            objs = sorted(objs, key=lambda o: float(o['flux']), reverse=True)[:max_stars]

    psf_list = []
    for o in objs:
        res = fit_psf_on_patch(sub, float(o['x']), float(o['y']), half_size=half_size, hr_scale=1)
        if res is not None and np.isfinite(res['fwhm']):
            res['flux'] = float(o['flux'])
            psf_list.append(res)

    return psf_list


def filter_outliers_iqr(values, k=1.5, min_points=8):
    """Remove outliers from `values` using the standard IQR rule
    (keep values within [Q1 - k*IQR, Q3 + k*IQR]).
    Returns (cleaned_array, n_removed, low_bound, high_bound).
    If there are too few points, returns the values unfiltered."""
    arr = np.asarray(values, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return arr, 0, np.nan, np.nan

    if arr.size < min_points:
        return arr, 0, np.nan, np.nan

    q1, q3 = np.percentile(arr, [25.0, 75.0])
    iqr = q3 - q1
    if (not np.isfinite(iqr)) or (iqr <= 0):
        return arr, 0, np.nan, np.nan

    low = q1 - k * iqr
    high = q3 + k * iqr
    keep = (arr >= low) & (arr <= high)
    cleaned = arr[keep]

    removed = int(arr.size - cleaned.size)
    if cleaned.size == 0:
        return arr, 0, low, high

    return cleaned, removed, low, high


def plot_fit_profiles(fig, spec, psf_fit):
    """Draw the X and Y intensity-profile plots (data vs. Gaussian fit)
    for the brightest star's PSF fit, into the given gridspec `spec`."""
    ax_x = fig.add_subplot(spec[0, 0])
    ax_y = fig.add_subplot(spec[1, 0])

    if psf_fit is None:
        for ax, label in ((ax_x, 'X profile'), (ax_y, 'Y profile')):
            ax.text(0.2, 0.5, 'No valid bright-star fit', transform=ax.transAxes)
            ax.set_title(label)
            ax.set_axis_off()
        return

    patch = psf_fit['patch']
    model = psf_fit.get('model')
    model_hr = psf_fit.get('model_hr')
    hr_scale = int(psf_fit.get('hr_scale', 1))

    row_idx = int(np.clip(round(psf_fit['y0_local']), 0, patch.shape[0] - 1))
    col_idx = int(np.clip(round(psf_fit['x0_local']), 0, patch.shape[1] - 1))

    x_coords = np.arange(patch.shape[1], dtype=np.float64)
    y_coords = np.arange(patch.shape[0], dtype=np.float64)

    ax_x.plot(x_coords, patch[row_idx, :], 'o', color='black', markersize=3, label='Data')
    if model_hr is not None and hr_scale > 1:
        row_hr = int(np.clip(round(psf_fit['y0_local'] * hr_scale), 0, model_hr.shape[0] - 1))
        x_hr_coords = np.linspace(0, patch.shape[1] - 1, patch.shape[1] * hr_scale)
        ax_x.plot(x_hr_coords, model_hr[row_hr, :], '-', color='crimson', linewidth=1.8, label='Fit')
    else:
        ax_x.plot(x_coords, model[row_idx, :], '-', color='crimson', linewidth=1.8, label='Fit')
    ax_x.set_title(f'X profile @ y={row_idx}')
    ax_x.set_xlabel('Patch X')
    ax_x.set_ylabel('Intensity')
    ax_x.grid(alpha=0.2)
    ax_x.legend(fontsize=8)

    ax_y.plot(y_coords, patch[:, col_idx], 'o', color='black', markersize=3, label='Data')
    if model_hr is not None and hr_scale > 1:
        col_hr = int(np.clip(round(psf_fit['x0_local'] * hr_scale), 0, model_hr.shape[1] - 1))
        y_hr_coords = np.linspace(0, patch.shape[0] - 1, patch.shape[0] * hr_scale)
        ax_y.plot(y_hr_coords, model_hr[:, col_hr], '-', color='royalblue', linewidth=1.8, label='Fit')
    else:
        ax_y.plot(y_coords, model[:, col_idx], '-', color='royalblue', linewidth=1.8, label='Fit')
    ax_y.set_title(f'Y profile @ x={col_idx}')
    ax_y.set_xlabel('Patch Y')
    ax_y.set_ylabel('Intensity')
    ax_y.grid(alpha=0.2)
    ax_y.legend(fontsize=8)


fits_root, stack_root = get_roots()
pdf_path = stack_root / 'pan_pair_triptych.pdf'
# Find every PAN1 frame; its matching PAN2 frame is derived by name substitution.
pan1_files = sorted(fits_root.rglob('*PAN1.fits')) if fits_root.exists() else []

combined = 0
missing = 0
failed = 0
cropped = 0
fallback_used = 0
all_psf_values = []   # collects every fitted FWHM across all pairs, for the final histogram

with PdfPages(pdf_path) as pdf:
    for pan1_path in pan1_files:
        pan2_name = re.sub(r'PAN1\\.fits$', 'PAN2.fits', pan1_path.name, flags=re.IGNORECASE)
        pan2_path = pan1_path.with_name(pan2_name)

        if not pan2_path.exists():
            print(f'MISSING PAIR: {pan1_path.relative_to(fits_root)} (no PAN2)')
            missing += 1
            continue

        try:
            data1 = fits.getdata(pan1_path).astype(np.float32)
            data2 = fits.getdata(pan2_path).astype(np.float32)
            if data1.ndim < 2 or data2.ndim < 2:
                print(f'INVALID DIMENSION: {pan1_path.name} or {pan2_path.name}')
                failed += 1
                continue

            if data1.shape[:2] != data2.shape[:2]:
                cropped += 1
                print(f'CROP TO OVERLAP: {pan1_path.name} {data1.shape[:2]} & {pan2_path.name} {data2.shape[:2]}')

            # Align both frames to a common center-cropped size before correlating.
            d1, d2 = crop_pair_to_common_center(data1, data2)

            shift_yx, corr_value, phase_err = estimate_shift_correlation(d1, d2)
            if not np.isfinite(corr_value):
                shift_yx = fallback_shift
                corr_value = -1.0
                fallback_used += 1

            stacked = stack_pair(d1, d2, shift_yx)

            # Measure PSF for all detected stars in the stacked image (default behavior).
            psf_list = measure_psf_for_bright_stars(stacked, measure_all=True, max_stars=max_labels, half_size=6)

            # Recompute a high-resolution model only for the brightest star (for plotting),
            # to avoid storing huge arrays for every star.
            brightest_psf = None
            if psf_list:
                brightest = max(psf_list, key=lambda p: p.get('flux', -np.inf))
                # Re-run fit with hr_scale=10 to obtain `model_hr` for plotting.
                # Use the same background-subtracted `sub` image to refit.
                finite = np.isfinite(stacked)
                fill = float(np.nanmedian(stacked[finite]))
                img = np.where(finite, stacked, fill).astype(np.float32)
                bkg = sep.Background(img)
                sub = img - bkg
                brightest_hr = fit_psf_on_patch(sub, brightest['x'], brightest['y'], half_size=6, hr_scale=10)
                if brightest_hr is not None:
                    brightest_hr['flux'] = brightest.get('flux', brightest_hr.get('flux', np.nan))
                    brightest_psf = brightest_hr
                else:
                    brightest_psf = brightest

            # collect every star's FWHM into the global list (flattened across images)
            all_psf_values.extend([p['fwhm'] for p in psf_list if np.isfinite(p['fwhm'])])

            out_name = re.sub(r'PAN1\\.fits$', 'PAN_stack.fits', pan1_path.name, flags=re.IGNORECASE)
            out_path = stack_root / pan1_path.relative_to(fits_root).parent / out_name
            out_path.parent.mkdir(parents=True, exist_ok=True)

            # Copy the PAN1 header and record alignment/PSF metadata.
            hdr = fits.getheader(pan1_path)
            hdr['HISTORY'] = 'PAN2 aligned by correlation-only estimator; PSF measured by Gaussian fit (per-star)'
            hdr['SHIFT_Y'] = (shift_yx[0], 'Shift applied to PAN2 along Y (pixel)')
            hdr['SHIFT_X'] = (shift_yx[1], 'Shift applied to PAN2 along X (pixel)')
            hdr['SHFMETH'] = ('correlation', 'Shift method')
            hdr['CORRVAL'] = (float(corr_value), 'Normalized correlation score')
            hdr['PHERR'] = (float(phase_err) if np.isfinite(phase_err) else -1.0, 'Phase correlation error')
            hdr['NPSF'] = (len(psf_list), 'Number of bright stars with Gaussian PSF fit')
            hdr['STACKED'] = ('PAN1+PAN2', 'Two-frame stacked product')
            fits.PrimaryHDU(stacked.astype(np.float32), header=hdr).writeto(out_path, overwrite=True)

            p1_vmin, p1_vmax = image_limits(d1)
            p2_vmin, p2_vmax = image_limits(d2)
            st_vmin, st_vmax = image_limits(stacked)

            # Build a 2x2 page: PAN1, PAN2, STACK+PSF markers, and fit profiles.
            fig = plt.figure(figsize=(12, 8))
            grid = fig.add_gridspec(2, 2)
            ax1 = fig.add_subplot(grid[0, 0])
            ax2 = fig.add_subplot(grid[0, 1])
            ax3 = fig.add_subplot(grid[1, 0])
            profile_spec = grid[1, 1].subgridspec(2, 1, hspace=0.4)

            ax1.imshow(d1, origin='lower', cmap='gray', vmin=p1_vmin, vmax=p1_vmax)
            ax1.set_title('PAN1')
            ax2.imshow(d2, origin='lower', cmap='gray', vmin=p2_vmin, vmax=p2_vmax)
            ax2.set_title('PAN2')
            ax3.imshow(stacked, origin='lower', cmap='gray', vmin=st_vmin, vmax=st_vmax)
            ax3.set_title('STACK + PSF')

            # Mark each fitted star and label it with its measured FWHM.
            for p in psf_list:
                ax3.plot(p['x'], p['y'], marker='o', markersize=4, markerfacecolor='none', markeredgecolor='cyan', linewidth=0.8)
                ax3.text(p['x'] + 0.8, p['y'] + 0.8, f"PSF={p['fwhm']:.2f}px", color='yellow', fontsize=7)

            for ax in (ax1, ax2, ax3):
                ax.set_xlabel('X')
                ax.set_ylabel('Y')

            plot_fit_profiles(fig, profile_spec, brightest_psf)

            median_psf = float(np.median([p['fwhm'] for p in psf_list])) if psf_list else float('nan')
            fig.suptitle(f"{pan1_path.stem} | shift=(dy={shift_yx[0]:.3f}, dx={shift_yx[1]:.3f}), corr={corr_value:.4f}, psf_n={len(psf_list)}, psf_med={median_psf:.2f}px")
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

            print(
                f'STACKED: {pan1_path.relative_to(fits_root)} + {pan2_path.relative_to(fits_root)} '
                f'| shift={shift_yx}, corr={corr_value:.4f}, psf_n={len(psf_list)}, psf_med={median_psf:.2f} '
                f'-> {out_path.relative_to(stack_root.parent)}'
            )
            combined += 1

        except Exception as exc:
            print(f'FAILED: {pan1_path.name} ({exc})')
            failed += 1

    # Final chart: PSF distribution across all stacked images (outliers removed by IQR rule).
    if len(all_psf_values) > 0:
        fwhm_raw = np.asarray(all_psf_values, dtype=np.float64)
        fwhm_clean, removed_n, low, high = filter_outliers_iqr(
            fwhm_raw,
            k=outlier_iqr_k,
            min_points=min_points_for_iqr,
        )

        medv = float(np.median(fwhm_clean))
        meanv = float(np.mean(fwhm_clean))

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(fwhm_clean, bins=15, color='#4C78A8', edgecolor='white', alpha=0.9)
        ax.axvline(medv, color='orange', linestyle='--', linewidth=1.8, label=f'Median={medv:.2f}px')
        ax.axvline(meanv, color='crimson', linestyle='-.', linewidth=1.8, label=f'Mean={meanv:.2f}px')
        ax.set_title('PSF FWHM Distribution (Outliers Removed)')
        ax.set_xlabel('FWHM [pixel]')
        ax.set_ylabel('Count')
        ax.legend()
        ax.grid(alpha=0.2)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        print(
            f'PSF Distribution (cleaned): N={len(fwhm_clean)}, removed={removed_n}, '
            f'median={medv:.2f}px, mean={meanv:.2f}px'
        )
        if np.isfinite(low) and np.isfinite(high):
            print(f'IQR bounds: [{low:.2f}, {high:.2f}] px')
    else:
        print('PSF Distribution: no valid PSF measurements to plot')

print(f'PDF: {pdf_path}')
print(f'Summary: {combined} stacked, {missing} missing pairs, {failed} failed, {cropped} overlap-cropped, {fallback_used} fallback-used')


STACKED: 20260525_PAN1.fits + 20260525_PAN1.fits | shift=(-1.1102230246251565e-16, -1.1102230246251565e-16), corr=1.0000, psf_n=2, psf_med=1.69 -> fits_stacked/20260525_PAN1.fits
STACKED: 20260602_PAN1.fits + 20260602_PAN1.fits | shift=(-1.1102230246251565e-16, -1.1102230246251565e-16), corr=1.0000, psf_n=3, psf_med=1.60 -> fits_stacked/20260602_PAN1.fits
STACKED: 20260603_PAN1.fits + 20260603_PAN1.fits | shift=(-1.1102230246251565e-16, -1.1102230246251565e-16), corr=1.0000, psf_n=3, psf_med=1.62 -> fits_stacked/20260603_PAN1.fits
STACKED: 20260622_PAN1.fits + 20260622_PAN1.fits | shift=(-1.1102230246251565e-16, -1.1102230246251565e-16), corr=1.0000, psf_n=2, psf_med=3.11 -> fits_stacked/20260622_PAN1.fits
STACKED: 20260702_PAN1.fits + 20260702_PAN1.fits | shift=(-1.1102230246251565e-16, -1.1102230246251565e-16), corr=1.0000, psf_n=2, psf_med=1.72 -> fits_stacked/20260702_PAN1.fits
STACKED: 20260707_PAN1.fits + 20260707_PAN1.fits | shift=(-1.1102230246251565e-16, -1.1102230246251565e-1